# CipherMark -- Config 2/3 : entrainement 256 bits sur Colab

Ce notebook reproduit `configs/training/ciphermark/posthoc_pixel_256bits_colab.yaml` :
resolution reelle (256), `nbits=256` (au lieu des 64 par defaut de DistSeal), corpus
elargi et representatif (COCO + Kodak + BSDS + scikit-image).

**A faire avant de lancer** : `Runtime > Change runtime type > GPU (T4 ou mieux)`.

**Reprise entre sessions** : le checkpoint est sauve sur Google Drive quand le
mount reussit (`output_dir` pointe vers `/content/drive/...`). Si la session
Colab se coupe (deconnexion ~12h en gratuit), remontez Drive et relancez
simplement la cellule "6b. Entrainement" telle quelle -- `train.py` reprend
automatiquement depuis `checkpoint.pth` (train.py:440-450), sans rien a changer.

## 1. Monter Google Drive (avec repli automatique si ca echoue)

Erreur frequente : `MessageError: Error: credential propagation was unsuccessful`.
C'est un probleme cote navigateur, pas le notebook. Avant de relancer cette
cellule :

1. Autoriser les cookies tiers pour `google.com` / `colab.research.google.com`
   (icone cadenas dans la barre d'adresse > Parametres du site > Cookies).
2. Ne pas etre en navigation privee/Incognito.
3. Desactiver temporairement les bloqueurs de pub / extensions (uBlock, etc.).
4. `Runtime > Restart session`, puis relancer cette cellule en premier.
5. Se deconnecter/reconnecter au compte Google dans le navigateur.

Si ca persiste malgre tout, la cellule bascule automatiquement sur un
stockage local (ephemere : perdu a la deconnexion) pour ne pas bloquer le
travail -- pensez alors a telecharger le checkpoint manuellement avec la
derniere cellule du notebook.

In [ ]:
import os
from google.colab import drive

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ciphermark/runs/colab_256bits"
LOCAL_OUTPUT_DIR = "/content/runs/colab_256bits"

try:
    drive.mount('/content/drive', force_remount=True)
    OUTPUT_DIR = DRIVE_OUTPUT_DIR
    print("Drive monte -- checkpoint persistant entre sessions :", OUTPUT_DIR)
except Exception as e:
    print("Mount Drive echoue ({}) -- on continue SANS Drive.".format(e))
    print("Checkpoint EPHEMERE pour cette session -- voir la cellule de troubleshooting ci-dessus,")
    print("et pensez a telecharger le checkpoint manuellement en fin de run (derniere cellule).")
    OUTPUT_DIR = LOCAL_OUTPUT_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR =", OUTPUT_DIR)

## 2. Cloner (ou mettre a jour) le depot
Branche `ciphermark`, qui contient les correctifs nbits=256 (sync CUDA, VideoWam,
attenuation None) et les configs de ce notebook.

In [ ]:
REPO_URL = "https://github.com/ngueagho/distseal-meta.git"
REPO_DIR = "code-memoire"

if not os.path.exists(REPO_DIR):
    !git clone -b ciphermark {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Installer les dependances

In [ ]:
!pip install -q -r requirements.txt

## 4. Construire le corpus (reseau datacenter Colab -- pas la connexion locale)
Le meme telechargement plante/rampe en local (COCO ~19 Go) ; il passe normalement
sans probleme depuis Colab. Ajuster `COCO_MAX` a la baisse si le telechargement
est trop lent sur la session du jour.

In [ ]:
COCO_MAX = 5000

!python -m scripts.ciphermark.build_corpus --out corpus-colab-raw \
    --coco unlabeled2017 --coco-max {COCO_MAX} --kodak --bsds

## 5. Split train / val (90 / 10)

In [ ]:
import shutil

src = "corpus-colab-raw"
files = sorted(os.listdir(src))
os.makedirs("corpus-colab/train", exist_ok=True)
os.makedirs("corpus-colab/val", exist_ok=True)

for i, f in enumerate(files):
    dst = "corpus-colab/val" if i < len(files) // 10 else "corpus-colab/train"
    shutil.copy(os.path.join(src, f), os.path.join(dst, f))

print("train:", len(os.listdir("corpus-colab/train")))
print("val:  ", len(os.listdir("corpus-colab/val")))

## 6a. Calibration (5 min) -- a lancer avant le run long
Regarder la valeur `s / it` dans les logs pour estimer le temps reel des 20 000
pas prevus par la config, et ajuster `epochs`/`iter_per_epoch` ci-dessous si besoin.

In [ ]:
!torchrun --nproc_per_node=1 --standalone train.py \
    --config configs/training/ciphermark/posthoc_pixel_256bits_colab.yaml \
    --output_dir {OUTPUT_DIR}_calib \
    --epochs 2 --iter_per_epoch 20

## 6b. Le run pour de vrai
Reexecuter cette cellule telle quelle apres une deconnexion (et apres avoir
remonte Drive via la cellule 1) : reprise automatique depuis
`{OUTPUT_DIR}/checkpoint.pth`.

In [ ]:
!torchrun --nproc_per_node=1 --standalone train.py \
    --config configs/training/ciphermark/posthoc_pixel_256bits_colab.yaml \
    --output_dir {OUTPUT_DIR}

## 7. Si vous n'avez PAS pu monter Drive : telecharger le checkpoint manuellement
A executer avant de fermer la session, sinon le travail est perdu (stockage
ephemere `/content/...`).

In [ ]:
from google.colab import files

ckpt_path = f"{OUTPUT_DIR}/checkpoint.pth"
if OUTPUT_DIR == LOCAL_OUTPUT_DIR and os.path.exists(ckpt_path):
    files.download(ckpt_path)
else:
    print("Rien a telecharger : soit Drive est monte (checkpoint deja persistant),",
          "soit le checkpoint n'existe pas encore a", ckpt_path)